# 05 — Target encoding SANS FUITE + TE du couple

Deux changements vs notebook 04 :
1. **TE sans fuite** : les lignes d'entraînement sont encodées en OOF imbriqué (jamais par
   elles-mêmes). -> le gain mesuré est honnête, comparable au LB.
2. On ajoute le **TE du couple (émetteur, destinataire)**.

Objectif : pousser le **last fold à ~0.365+** (=> LB visé > 0.35). On ne soumet QUE si la CV honnête le justifie.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv")
op03 = op03_mask(train).to_numpy()
y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))

In [ ]:
EPS = 1e-6
PAIR = "pair"

def with_pair(df):
    return df.assign(**{PAIR: df[C.ORIGIN_ACCT].astype(str) + '||' + df[C.DEST_ACCT].astype(str)})

def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X

def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    return pd.concat([X, beh, rec], axis=1)

def make_model():
    try:
        from catboost import CatBoostClassifier
        return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                                  learning_rate=0.05, iterations=600, random_seed=42, verbose=False)
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, random_state=42)

TE_COLS = [C.ORIGIN_ACCT, C.DEST_ACCT, PAIR]

def te_train(X, ref_p, te_cols):
    """TE OOF imbriqué pour les lignes d'entraînement (ref_p == ces lignes, avec pair+label)."""
    X = X.copy()
    for col in te_cols:
        X[f"te_{col}"] = oof_target_encode_train(ref_p, col, C.TARGET)
    return X

def te_apply(X, df_p, ref_p, te_cols):
    """TE pour valid/test : table apprise sur ref_p (passé), appliquée à df_p."""
    X = X.copy()
    for col in te_cols:
        mp, gm = fit_target_map(ref_p, col, C.TARGET)
        X[f"te_{col}"] = apply_target_map(df_p, col, mp, gm)
    return X

## A/B honnête : sans TE vs TE sans fuite (émetteur+destinataire+couple)

In [ ]:
def run_cv(te_cols):
    oof = np.zeros(len(train)); per_fold = []; lm = lc = None
    for tr_idx, va_idx in folds_full:
        tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]
        ref = train.iloc[tr_op]; ref_p = with_pair(ref)
        Xtr = base_build(train.iloc[tr_op], ref)
        Xva = base_build(train.iloc[va_op], ref)
        if te_cols:
            Xtr = te_train(Xtr, ref_p, te_cols)
            Xva = te_apply(Xva, with_pair(train.iloc[va_op]), ref_p, te_cols)
        m = make_model(); m.fit(Xtr, y_all[tr_op])
        oof[va_op] = m.predict_proba(Xva)[:, 1]
        per_fold.append(evaluate_ap(y_all[va_op], oof[va_op])); lm, lc = m, Xtr.columns
    return per_fold, oof, lm, lc

pf_base, _, _, _ = run_cv(te_cols=[])
pf_te, oof_te, m_te, c_te = run_cv(te_cols=TE_COLS)

def show(name, pf):
    print(f"{name:24s} global {np.mean(pf):.4f} | recent(2) {np.mean(pf[-2:]):.4f} | last {pf[-1]:.4f}")
show("sans TE", pf_base)
show("+ TE sans fuite", pf_te)
print("\nper-fold avec TE :", [round(x, 4) for x in pf_te])
print("Gain last fold   :", round(pf_te[-1] - pf_base[-1], 4))
print("LB estimé (last - 0.015) :", round(pf_te[-1] - 0.015, 4))

In [ ]:
imp = m_te.get_feature_importance() if hasattr(m_te, "get_feature_importance") else m_te.feature_importances_
print(pd.Series(imp, index=c_te).sort_values(ascending=False).round(2).head(15))

## Soumission (à n'exécuter QUE si `LB estimé` > 0.35)

In [ ]:
from src.calibration import fit_isotonic, apply_isotonic
from src.utils import make_submission
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")

ref_full = train.iloc[np.where(op03)[0]]; ref_full_p = with_pair(ref_full)
Xf = te_train(base_build(ref_full, ref_full), ref_full_p, TE_COLS)
final = make_model(); final.fit(Xf, y_all[op03])
iso = fit_isotonic(oof_te[op03], y_all[op03])

te_op = op03_mask(test).to_numpy()
test_op = test.iloc[np.where(te_op)[0]]
Xte = te_apply(base_build(test_op, ref_full), with_pair(test_op), ref_full_p, TE_COLS)
proba = apply_isotonic(iso, final.predict_proba(Xte)[:, 1])
full = np.zeros(len(test)); full[te_op] = proba
path = make_submission(test[C.ID], full, "05_te_clean")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))